In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/generation_config.json
/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md


!pip install -q -U kaggle-benchmarks

## Democratic Hammurabi Benchmark Task Definition

In [3]:
import kaggle_benchmarks as kbench
import random
import math
import os

class DemocraticHammurabi:
    def __init__(self, max_years=12):
        self.max_years = max_years
        self.reset()

    def reset(self):
        self.year = 1
        self.population = 100
        self.grain = 2800
        self.land = 1000
        self.land_price = random.randint(17, 26)

        # Faction approvals (0 to 100)
        self.farmers_approval = 50.0
        self.workers_approval = 50.0
        self.elites_approval = 50.0

        self.is_done = False
        self.game_over_reason = ""
        self.starved_total = 0

        return self._get_state()

    def _get_state(self):
        years_until_election = 4 - (self.year % 4)
        if years_until_election == 4:
            # If year % 4 == 0, the election is THIS year.
            years_until_election = 0

        return [
            self.year,
            self.population,
            self.grain,
            self.land,
            self.land_price,
            self.farmers_approval,
            self.workers_approval,
            self.elites_approval,
            years_until_election
        ]

    def step(self, actions):
        """
        Actions should be a list of 3 continuous values:
        - action_land: [-1, 1]. <0 means sell fraction of land, >0 means buy fraction of max affordable land.
        - action_feed: [0, 1]. fraction of total grain to use for feeding people.
        - action_plant: [0, 1]. fraction of remaining grain to use for planting.
        """
        if self.is_done:
            return self._get_state(), 0, self.is_done, {"reason": "Already done"}

        action_land, action_feed, action_plant = actions

        # Clip actions to expected ranges
        action_land = max(-1.0, min(1.0, action_land))
        action_feed = max(0.0, min(1.0, action_feed))
        action_plant = max(0.0, min(1.0, action_plant))

        # 1. Buy or Sell Land
        land_changed = 0
        if action_land < 0:
            # Sell land
            acres_to_sell = int(abs(action_land) * self.land)
            self.land -= acres_to_sell
            self.grain += acres_to_sell * self.land_price
            land_changed = -acres_to_sell
        elif action_land > 0:
            # Buy land
            max_acres_affordable = self.grain // self.land_price
            acres_to_buy = int(action_land * max_acres_affordable)
            self.land += acres_to_buy
            self.grain -= acres_to_buy * self.land_price
            land_changed = acres_to_buy

        # 2. Feed People
        grain_for_food = int(action_feed * self.grain)
        self.grain -= grain_for_food

        people_fed = grain_for_food // 20
        starved = max(0, self.population - people_fed)
        self.starved_total += starved

        if starved > 0:
            self.population -= starved

        # Immediate game over if starvation is extreme (>45%)
        if starved > 0.45 * (self.population + starved):
            self.is_done = True
            self.game_over_reason = "Impeached for extreme starvation"
            return self._get_state(), self._calculate_reward(), self.is_done, {"reason": self.game_over_reason}

        # 3. Plant Seeds
        grain_for_planting = int(action_plant * self.grain)
        max_plantable_by_people = self.population * 10

        # You need 1 grain per acre
        actual_planted = min(grain_for_planting, self.land, max_plantable_by_people)
        self.grain -= actual_planted

        # 4. Harvest & Rats
        yield_per_acre = random.randint(1, 5)
        harvest = actual_planted * yield_per_acre
        self.grain += harvest

        rats_ate = 0
        if random.random() < 0.4: # 40% chance of rats
            rats_ate = int(self.grain * random.uniform(0.1, 0.3))
            self.grain -= rats_ate

        # 5. Demographics (Births & Immigrants)
        immigrants = 0
        if starved == 0:
            immigrants = random.randint(1, 10) + int((20 * self.land + self.grain) / (100 * self.population + 1))
            self.population += immigrants

        # 6. Faction Approval Updates
        # Farmers: Like buying land and planting. Hate selling land.
        if land_changed > 0:
            self.farmers_approval += 5
        elif land_changed < 0:
            self.farmers_approval -= 10
        if actual_planted == self.land:
            self.farmers_approval += 5

        # Workers: Hate starvation, like food surplus/low prices.
        if starved > 0:
            self.workers_approval -= (starved / self.population) * 100
        else:
            self.workers_approval += 5

        # Elites: Like wealth accumulation
        total_wealth = self.grain + (self.land * self.land_price)
        expected_wealth = 2800 + (1000 * 20) # Baseline
        if total_wealth > expected_wealth:
            self.elites_approval += 5
        else:
            self.elites_approval -= 5

        # Decay approvals towards 50
        self.farmers_approval = self._clamp_and_decay_approval(self.farmers_approval)
        self.workers_approval = self._clamp_and_decay_approval(self.workers_approval)
        self.elites_approval = self._clamp_and_decay_approval(self.elites_approval)

        # 7. Elections (Every 4 years)
        if self.year % 4 == 0:
            average_approval = (self.farmers_approval + self.workers_approval + self.elites_approval) / 3.0
            if average_approval < 45.0:
                self.is_done = True
                self.game_over_reason = f"Lost election with {average_approval:.1f}% approval"
                return self._get_state(), self._calculate_reward(), self.is_done, {"reason": self.game_over_reason}

        # 8. End of Year Updates
        self.year += 1
        self.land_price = random.randint(17, 26)

        if self.year > self.max_years:
            self.is_done = True
            self.game_over_reason = "Completed term successfully!"

        return self._get_state(), self._calculate_reward(), self.is_done, {"reason": self.game_over_reason}

    def _clamp_and_decay_approval(self, approval):
        # Move slightly towards 50
        if approval > 50:
            approval -= (approval - 50) * 0.1
        elif approval < 50:
            approval += (50 - approval) * 0.1
        return max(0.0, min(100.0, approval))

    def _calculate_reward(self):
        # 1. Survival Bonus
        survival_bonus = self.year * 100

        # 2. Wealth Score (Fixed Loophole: Land and Grain are roughly equivalent in value)
        # Assuming avg land price is ~20, an acre is worth 20 grain.
        # We scale it down so the numbers don't overwhelm other metrics.
        wealth_score = (self.land * 20 + self.grain) / 50.0

        # 3. Population Score
        pop_score = self.population * 2

        # 4. Approval Score (Fixed Loophole: Agent is scored on the most unhappy faction)
        # Using min() forces the agent to balance all three factions.
        # If any faction drops to 0, they get 0 approval points.
        approval_score = min(self.farmers_approval, self.workers_approval, self.elites_approval) * 3.0

        # 5. Starvation Penalty
        penalty = self.starved_total * 50

        total_score = survival_bonus + wealth_score + pop_score + approval_score - penalty

        # 6. Impeachment Penalty (Fixed Loophole: Slash score drastically instead of flat penalty)
        if self.is_done and self.year <= self.max_years:
            # If they fail early, their entire score is divided by 10.
            # This ruins any hoarded wealth advantage.
            total_score = total_score / 10.0

        return total_score


class LLMDemocraticHammurabi:
    def __init__(self, max_years=12):
        self.env = DemocraticHammurabi(max_years=max_years)
        self.state = self.env.reset()

    def check_decree(self, acres_to_buy, bushels_to_feed, acres_to_plant):
        year, pop, grain, land, land_price, f_app, w_app, e_app, yrs_to_elec = self.state

        cost_of_land = acres_to_buy * land_price if acres_to_buy > 0 else 0
        revenue_from_land = abs(acres_to_buy) * land_price if acres_to_buy < 0 else 0
        cost_of_planting = acres_to_plant * 1.0

        total_costs = cost_of_land + bushels_to_feed + cost_of_planting
        available_funds = grain + revenue_from_land
        projected_bushels = available_funds - total_costs

        if projected_bushels < 0:
            return False, f"ACCOUNTANT ERROR: Sire, your decree costs {total_costs} bushels. We only have {available_funds} available. We are short by {abs(projected_bushels)} bushels."

        projected_acres = land + acres_to_buy
        if projected_acres < acres_to_plant:
            return False, f"ACCOUNTANT ERROR: Sire, you ordered us to plant {acres_to_plant} acres, but we only own {projected_acres} acres of land!"

        max_plantable = pop * 10
        if acres_to_plant > max_plantable:
            return False, f"ACCOUNTANT ERROR: Sire, our {pop} people can only plant a maximum of {max_plantable} acres, but you ordered {acres_to_plant}."

        return True, "The math is sound, Sire."

    def play_turn(self, acres_to_buy, bushels_to_feed, acres_to_plant):
        year, pop, grain, land, land_price, f_app, w_app, e_app, yrs_to_elec = self.state

        if acres_to_buy < 0:
            action_land = acres_to_buy / max(1.0, float(land))
        elif acres_to_buy > 0:
            max_affordable = (grain) // land_price
            action_land = acres_to_buy / max(1.0, float(max_affordable))
        else:
            action_land = 0.0

        projected_grain = grain
        if acres_to_buy < 0:
            projected_grain += abs(acres_to_buy) * land_price
        elif acres_to_buy > 0:
            projected_grain -= acres_to_buy * land_price

        action_feed = bushels_to_feed / max(1.0, float(projected_grain)) if projected_grain > 0 else 0.0
        projected_grain -= bushels_to_feed

        action_plant = acres_to_plant / max(1.0, float(projected_grain)) if projected_grain > 0 else 0.0

        actions = [action_land, action_feed, action_plant]

        self.state, reward, done, info = self.env.step(actions)
        return done, info['reason']

    def get_slm_payload(self):
        year, pop, grain, land, land_price, f_app, w_app, e_app, yrs_to_elec = self.state
        optimal_food = pop * 20

        return f"""
        REPORT:
        Year: {year}
        Years Until Next Election: {yrs_to_elec}

        RESOURCES:
        Population: {pop}
        Acres of Land: {land}
        Bushels in Storage: {grain}
        Current Land Price: {land_price} bushels/acre
        (NOTE: Your people require {optimal_food} bushels to avoid starvation this year).

        POLITICAL POLLS (Must stay above 45% average to win election):
        Farmers Approval: {f_app:.1f}%
        Workers Approval: {w_app:.1f}%
        Elites Approval: {e_app:.1f}%
        """



You are the Grand Vizier of Democratic Babylon.
I will provide you with a 'REPORT' containing raw data about the kingdom.
1. Think silently about the implications of the data, your budget, and the political polls.
2. Calculate your budget based on the Laws of Babylon.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.
4. You must output your tool call using EXACTLY this syntax, replacing the values with your calculated numbers:
<|tool_call>call:issue_decree{acres_to_buy: [number], bushels_to_feed: [number], acres_to_plant: [number]}<tool_call|>

THE LAWS OF BABYLON (GAME MECHANICS):
Before making your decrees, you MUST calculate your budget in your thought block using these exact rules:
1. FEEDING: 1 person requires exactly 20 bushels to survive the year. If you feed them less, people will starve. Starvation drastically lowers Worker approval and causes instant impeachment if too high!
2. PLANTING: It costs exactly 1 bushe

In [4]:

@kbench.task(name="democratic_hammurabi")
def play_democratic_hammurabi(
    llm: kbench.Actor,
    max_years: int = 12,
    max_retries_per_turn: int = 3
) -> None:
    """
    Evaluates an LLM's ability to act as the Grand Vizier of Democratic Babylon,
    balancing resources and political factions over multiple turns.
    """
    game = LLMDemocraticHammurabi(max_years=max_years)
    current_decree = []

    def issue_decree(acres_to_buy: int, bushels_to_feed: int, acres_to_plant: int) -> str:
        """
        Issues the royal decrees for the year, deciding the fate of Babylon.
        
        Args:
            acres_to_buy: The number of acres to buy (use a negative number to sell land).
            bushels_to_feed: The number of bushels to distribute to the populace for food.
            acres_to_plant: The number of acres to plant with seed for the next harvest.
        """
        valid, msg = game.check_decree(acres_to_buy, bushels_to_feed, acres_to_plant)
        if valid:
            current_decree.clear()
            current_decree.extend([acres_to_buy, bushels_to_feed, acres_to_plant])
            return "The math is sound, Sire. Executing Decrees..."
        else:
            return msg

    system_prompt = """You are the Grand Vizier of Democratic Babylon.
I will provide you with a 'REPORT' containing raw data about the kingdom.
1. Think silently about the implications of the data, your budget, and the political polls.
2. Calculate your budget based on the Laws of Babylon.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.

THE LAWS OF BABYLON (GAME MECHANICS):
Before making your decrees, you MUST calculate your budget in your thought block using these exact rules:
1. FEEDING: 1 person requires exactly 20 bushels to survive the year. If you feed them less, people will starve. Starvation drastically lowers Worker approval and causes instant impeachment if too high!
2. PLANTING: It costs exactly 1 bushel of seed to plant 1 acre of land. You cannot plant more acres than you own. 1 person can plant a maximum of 10 acres.
3. REAL ESTATE: Buying 1 acre costs the current 'Land Price' in bushels. Selling 1 acre (using a negative number for acres_to_buy) ADDS the 'Land Price' in bushels to your total available budget.
4. THE GOLDEN RULE: (Bushels spent on Buying Land) + (Bushels spent on Feeding) + (Bushels spent on Planting) MUST NOT exceed your total available Bushels (which includes any Bushels gained from Selling Land).

THE RULES OF POLITICS (HOW TO WIN):
1. ELECTIONS: An election occurs every 4 years. If your average approval drops below 45%, you will be impeached and lose the game.
2. FARMERS: Farmers love it when you buy land and plant seeds. They HATE it when you sell land.
3. WORKERS: Workers love when you have extra food, and they absolutely HATE starvation.
4. ELITES: Elites only care about total kingdom wealth. They want you to accumulate massive amounts of grain and land value.

You must balance these factions while surviving!"""
    
    kbench.system.send(system_prompt)
    
    done = False
    reason = ""
    
    while not done:
        report = "Current Kingdom Status:\n" + game.get_slm_payload()
        
        current_decree.clear()
        retries = 0
        prompt_text = report
        
        while len(current_decree) == 0:
            if retries >= max_retries_per_turn:
                done = True
                reason = "Impeached for analysis paralysis (failed to issue a valid decree after max retries)."
                break
                
            try:
                response = llm.prompt(prompt_text, tools=[issue_decree])
            except Exception as e:
                done = True
                reason = f"Impeached for analysis paralysis (LLM Error: {type(e).__name__})."
                break
            
            if len(current_decree) == 0:
                last_msg = kbench.chats.current().messages[-1]
                calls = getattr(last_msg, "tool_calls", [])
                
                if not calls and last_msg.sender != "tool":
                    prompt_text = "You did not issue a decree. You MUST call the `issue_decree` tool to end your turn."
                else:
                    prompt_text = "Please review the Accountant's error and recalculate your decree."
                retries += 1
            else:
                break
                
        if len(current_decree) == 3:
            buy, feed, plant = current_decree
            done, reason = game.play_turn(buy, feed, plant)
                
    survived = (game.env.year > max_years)
    
    # Encode stats directly into expectation since details is not supported
    kbench.assertions.assert_true(
        survived,
        expectation=f"Agent must survive 12 years. | Year: {game.env.year} | Starved: {game.env.starved_total} | Pop: {game.env.population} | Reason: {reason or game.env.game_over_reason}"
    )


In [ ]:

# Run the benchmark
result = play_democratic_hammurabi.run(
    llm=kbench.llm,
    max_years=12
)

print("\n--- Benchmark Results ---")
print(f"Status: {result.status.name}")
if result.assertion_results:
    assertion = result.assertion_results[0]
    print(f"Passed: {assertion.passed}")
    print(f"Details: {assertion.expectation}")
